# Export ONNX

Export the final deployment Keras model to ONNX and write deployment metadata.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import onnx
import tensorflow as tf
import tf2onnx

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())


In [ ]:
def export_model_to_onnx(model_path: Path, onnx_path: Path, opset: int = 13) -> Path:
    model = tf.keras.models.load_model(model_path, compile=False)
    if not hasattr(model, 'output_names'):
        model.output_names = [tensor.name.split(':')[0] for tensor in model.outputs]
    input_shape = tuple(int(dim) if dim is not None else None for dim in model.input_shape)
    ensure_dir(onnx_path.parent)
    tf2onnx.convert.from_keras(
        model,
        input_signature=(tf.TensorSpec(input_shape, tf.float32, name='image_input'),),
        opset=opset,
        output_path=str(onnx_path),
    )
    onnx.checker.check_model(onnx.load(str(onnx_path)))
    return onnx_path


def build_onnx_metadata(model_path: Path, base_metadata: dict[str, object]) -> dict[str, object]:
    model = tf.keras.models.load_model(model_path, compile=False)
    metadata = dict(base_metadata)
    metadata.setdefault('model_name', model.name)
    metadata.setdefault('backbone', 'MobileNetV3Small')
    metadata.setdefault('model_input_mode', 'cnn_only')
    metadata.setdefault('image_crop_mode', metadata.get('input_mode', INPUT_MODE))
    metadata['input_mode'] = metadata.get('input_mode', INPUT_MODE)
    metadata['input_shape'] = list(model.input_shape[1:])
    metadata['labels'] = metadata.get('labels', LABEL_ORDER)
    return metadata


In [ ]:
MODEL_PATH = Path(str(override('MODEL_PATH', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8samples_final_deployment_cnn_only' / 'models' / 'meatlens_final_8samples_cnn_only_mobilenetv3small.keras')))
ONNX_PATH = Path(str(override('ONNX_PATH', MODEL_PATH.with_suffix('.onnx'))))
ONNX_METADATA_PATH = Path(str(override('ONNX_METADATA_PATH', ONNX_PATH.with_suffix('.metadata.json'))))
BASE_METADATA_PATH = override('BASE_METADATA_PATH', None)
BASE_METADATA = dict(override('BASE_METADATA', {}))
if BASE_METADATA_PATH:
    BASE_METADATA = json.loads(Path(str(BASE_METADATA_PATH)).read_text(encoding='utf-8'))

export_model_to_onnx(MODEL_PATH, ONNX_PATH)
metadata = build_onnx_metadata(MODEL_PATH, BASE_METADATA)
ONNX_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'onnx_path: {ONNX_PATH}')
print(f'metadata_path: {ONNX_METADATA_PATH}')
